# Cluster Quality Diagnostic

**What this notebook answers.** You have an AlphaFold model. This shows how pLDDT is distributed across its protein family, where your protein sits in that distribution, and what the best and worst-predicted members of the cluster look like as structures and in alignment against yours.

**What it does not do.** It does not recommend that you switch models. AFDB clusters can contain distant homologues with different biological function, and no single available number reliably separates those from legitimate relatives, so the notebook shows you the evidence and leaves the judgement to you. See the closing section.

**To run it:** type a UniProt accession into the box below, then `Runtime` -> `Run all`.

In [ ]:
# Set up the environment. On Colab this clones the repository and puts it on the path.
import sys, subprocess
from pathlib import Path

REPO_BRANCH = "main"  # TODO(merge): pin to a release tag before release

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None

IN_COLAB = "google.colab" in sys.modules
REPO_ROOT = find_repo_root(Path.cwd())
if REPO_ROOT is None and IN_COLAB:
    target = Path("/content/InsightFold")
    if not (target / "src").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                        "https://github.com/<org>/InsightFold.git", str(target)], check=True)
    REPO_ROOT = target
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root (a directory holding pyproject.toml and src/).")

src = str(REPO_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print(f"Repository root: {REPO_ROOT}")

In [ ]:
%matplotlib inline
from insightfold import cluster_quality_utils as cq

cq.ensure_optional_packages()
cq.apply_notebook_style()

---
## 1. Your protein

<div style="border-left:4px solid #2152CE; padding-left:12px">
<b>Set the accession in the box below.</b> One bare UniProt accession, for example <code>P69905</code> (haemoglobin alpha), <code>Q13148</code> (TDP-43) or <code>Q9I1F6</code>. The <code>AF-…-F1</code> form is accepted and normalised. Isoforms such as <code>P04637-2</code> work too.
</div>

In [ ]:
#@title Choose a protein { display-mode: "form" }
#@markdown Enter a **UniProt accession**, then run all cells. The `AF-{ACC}-F1` form is accepted and normalised; isoforms such as `P04637-2` are accepted as they are.
ACCESSION = "O15552" #@param {type:"string"}
#@markdown ---
#@markdown If the protein named below is not the one you meant, stop and correct the accession.

ACCESSION = cq.resolve_accession(ACCESSION)
query_meta = cq.report_query(ACCESSION)

---
## 2. What the two clusterings are

AFDB groups its predictions two ways, and they answer different questions.

**AFDB50/MMseqs2** clusters by sequence: members share at most 50% identity with at least 90% bidirectional sequence overlap against the longest sequence of the cluster representative.

**AFDB/Foldseek** clusters by structure: an E-value below 0.01 with at least 90% bidirectional structural overlap against the largest structure of the representative.

These approximate what protein evolution calls a *family* and a *homologous superfamily*, in [InterPro's sense](https://www.ebi.ac.uk/training/online/courses/interpro-functional-and-structural-analysis/what-is-an-interpro-entry/interpro-entry-types/). They **approximate** those concepts; they are not curated InterPro entries.

If the sequence family cannot be analysed the notebook stops and says why. If only the structural group is unusable, that section alone is skipped and everything else still runs.

In [ ]:
# Large families can take a few seconds to arrive.
clusters = cq.load_clusters(ACCESSION)

---
## 3. Cluster summary

Everything below is computed from your protein's own cluster, whatever that cluster turns out to be. It describes the family as a whole. Where your protein sits inside it comes in the next section.

| Term                   | What it is                                                   | What to read from it                                         |
|------------------------|--------------------------------------------------------------|--------------------------------------------------------------|
| **Members**            | How many proteins AFDB cluster with yours                    | A handful means only a few proteins are homologous to your protein of interest. Tens of thousands means this protein space is widespread across evolution. |
| **Mean**               | The arithmetic average across all members                    | A single number for the family's overall level. A few very poor members drag it down, so read it next to the median. |
| **Median**             | The middle value: half the members score above it, half below | Usually the fairer summary of "a typical member". If the mean sits well below the median, the family has a long tail of poor models rather than being uniformly weak |
| **Standard deviation** | The typical distance between a member and the mean           | Small means the family is homogeneous and its members are predicted alike. Large means members genuinely disagree, and a single family-level number hides real variation. |
| **p25 to p75**         | The interquartile range: the width of the band containing the middle half of members. p25 is the 25th percentile, p75 the 75th | This is the most useful number here. When it is narrow, most of the family is packed into a couple of points, so ranking one member against another is not meaningful. When it is wide, position within the family carries real information. |
| **Confidence bands**   | The share of members falling in each of AFDB's four pLDDT bands | The shape of the family at a glance, and what a realistic ceiling looks like for this fold. |

Two things to hold on to while reading them.

**pLDDT is the predictor's own certainty, and nothing more.** It is not a measure of whether a protein will express, fold or crystallise, it has not been validated against experimental structures for these families, and every value here moves when AFDB is rebuilt.

**An average taken over a whole chain is substantially a measure of disorder content.** A family with a low mean may be predicted perfectly well across its ordered regions and simply carry long intrinsically disordered stretches. A low mean is not the same as a badly predicted family. The per-residue profiles further down are what separate the two.

In [ ]:
family = cq.report_family_summary(clusters)

### Your protein within the cluster

The line below gives your protein's own mean pLDDT and a **descriptor** placing it relative to the cluster summarised above.

The descriptor is derived from the cluster's own distribution rather than a fixed cutoff:

- **low extreme** below the 5th percentile of the cluster
- **high extreme** above the 95th percentile
- **typical** between the two

A descriptor rather than a rank, because how informative a percentile is depends on the dispersion of the cluster it is measured against. Where the interquartile range spans only a few pLDDT points, the difference between the 30th and the 70th percentile amounts to a fraction of a point and carries no practical information. The percentile is still reported, but interpret it against the interquartile range printed above.

If your protein is not a member of the cluster returned for it, no rank is defined and the notebook states that instead. This is a normal outcome, explained in the structural section below.


In [ ]:
position = cq.report_query_position(clusters)

In [ ]:
cq.show_family_distribution(clusters, family, position)

### Structurally similar proteins

Proteins grouped by structural similarity rather than sequence similarity. Structure is more conserved than sequence over evolutionary time, so Foldseek can detect remote homologues whose sequence identity has fallen below the threshold at which alignment-based methods can reliably infer homology.

**Your protein is usually not shown here.** The structure clustering is built from cluster representatives, so it is your family's representative that carries through rather than your specific entry. This figure describes the structural neighbourhood your protein sits in, not a ranking it takes part in, which is why it carries no marker for you.

These members typically score lower than the sequence cluster's, because a structural group spans much greater evolutionary distance.

Both rainclouds are drawn on **the same fixed pLDDT axis**, spanning the range covered by either cluster. Letting each figure choose its own axis would make two genuinely different distributions look alike, which is the opposite of what putting them together is for.

In [ ]:
cq.show_structure_cluster(clusters)

---
## 4. Length, and how it relates to confidence

Optional context. Members of a cluster are not all the same length, and length correlates with pLDDT, so this is worth seeing before reading anything into a single member's score. The joint density shows where the members actually sit, with marginal distributions on both axes and contours over the dense region.

In [ ]:
cq.show_length_vs_plddt(clusters)

---
## 5. Display preferences

<div style="border-left:4px solid #7FC9EF; padding-left:12px">
<b>Optional.</b> These affect only how the sections below are drawn.
</div>

**Model source** decides which of the two clusterings the best and worst models in the next section are taken from.

- `sequence` takes them from the sequence family. Its members are individual proteins, so the best and worst are real proteins.
- `structure` takes them from the structural group. That group is built from cluster representatives, so its extremes are the best member of a family rather than an individual protein.

Both are legitimate. They answer different questions.

In [ ]:
#@title Display preferences { display-mode: "form" }
#@markdown Optional. These affect only how the sections below are drawn.

MODEL_SOURCE_CLUSTER = "structure" #@param ["sequence", "structure"]
#@markdown - Which clustering the best and worst models are taken from. `sequence` gives individual proteins; `structure` gives family representatives.

ALIGNMENT_COLOUR_SCHEME = "clustal2" #@param ["clustal", "clustal2", "zappo", "taylor", "hydrophobicity", "helix", "strand", "turn", "buried", "flower", "blossom", "sunset", "ocean"]
#@markdown - Residue colours for the alignment panels. The confidence-band panels ignore this.

SHOW_PAE = True #@param {type:"boolean"}
#@markdown - Draw each model's predicted aligned error matrix beside its 3D view.

USER_MSA_PATH = "" #@param {type:"string"}
#@markdown - Optional path to your own a3m alignment. Leave empty to skip.

---
## 6. The best and worst-predicted members

**Before you look:** a cluster member can be a distant homologue with a different biological function, and this is expected rather than a data defect.

AFDB50 uses *cascaded* clustering. Members are linked to the final representative **transitively, through a chain of intermediate representatives**, each link within threshold. Identity to the representative itself, and between two arbitrary members, is therefore **unbounded below**.

A measured example: the best-scoring member of a human TDP-43 cluster is a yeast Prp24p protein. They align at 26.73% identity over 101 columns, covering **residues 80 to 171 of a 414-residue query**, which sits on TDP-43's RRM1 domain. Both are RRM-domain proteins, so this is remote homology, not an unrelated protein.

**No single number decides this.** Identity alone ranks that divergent case *above* a cluster of perfectly legitimate relatives (26.73% against 25.09%). Read the identity, the aligned range, the coverage, the description and the species together. The aligned range is the most stable of them; coverage shifts with alignment parameters.

This is a known property of cascaded clustering, not something this notebook discovered. See Steinegger and Söding 2018 (*Nat Commun* 9:2542) and Barrio-Hernandez et al. 2023 (*Nature* 621:637).

**Reading the 3D views.** Each model is coloured by AFDB's pLDDT confidence bands, the same colours as the raincloud above: dark blue above 90, light blue 70 to 90, yellow 50 to 70, orange below 50. Beside each structure is its **predicted aligned error** matrix. pLDDT tells you how confident the predictor is about each residue's local position; PAE tells you how confident it is about the relative position of every *pair* of residues, so a model can be locally confident everywhere and still have two domains whose arrangement is unresolved. Dark green is low error.

Set `MODEL_SOURCE_CLUSTER` above to draw these from the structure cluster instead.

Structures and matrices are downloaded here, several hundred kilobytes each. On a slow or unstable connection a download can be cut short, so each one is retried and checked for completeness before it is used. If a file cannot be retrieved after several attempts the notebook says so and carries on.

In [ ]:
models = cq.select_models(clusters, query_meta, MODEL_SOURCE_CLUSTER)

In [ ]:
cq.show_model_panels(models, show_pae=SHOW_PAE)

In [ ]:
cq.show_plddt_profiles(models)

---
## 7. How similar are they, really?

Local alignment against your protein, reported as identity **with the aligned block size, the coverage, and the aligned range**. Identity on its own is not interpretable: the same percentage over 40 columns of a 400-residue protein means something very different from the same percentage over 380.

In [ ]:
alignments = cq.report_alignments(models)

In [ ]:
cq.show_alignments(models, alignments, ALIGNMENT_COLOUR_SCHEME)

### Structural superposition

The alignment above pairs residues that look alike. This one pairs residues that are in the same place once the two structures are laid on top of one another, which is a different and stronger statement, and it is the one that still works when identity is low.

The superposition is computed with TM-align (Zhang and Skolnick 2005) and then handed to Mol\* to display. Mol\* does not compute the alignment: it receives the rotation and translation TM-align produced and applies them. Your protein is held fixed and the member is moved onto it.

Read the **TM-score** rather than the RMSD. TM-score is normalised to the length of your protein and runs from 0 to 1. Above 0.5 the two share a fold; below 0.3 is roughly what an unrelated pair of the same size would score. RMSD has no comparable scale, and TM-align maximises TM-score rather than minimising RMSD, so a fairly large RMSD across a well-aligned pair is ordinary and is not on its own a sign of failure.

In [ ]:
superpositions = cq.report_superpositions(models)

### How each model is presented below

Each model gets its own block: the superposition first, then the same alignment twice.

Both alignment panels show TM-align's own alignment, the one the superposition above them is built from, so a column means the two residues occupy the same position in space rather than merely resembling each other. Where this disagrees with the sequence alignment further up, it is the structural one that carries the evidence.

The first panel colours residues by chemistry, the second colours each residue by the confidence band of its own model. The second is the more useful one here: it shows whether a structural match rests on well-predicted residues on both sides, or whether one of the two is guessing.

Under each sequence is an **SSE** track, short for **secondary structure element**, read from the model file itself: `H` helix, `E` strand, `T` turn or bend. Residues in neither a helix, a strand nor a turn are left blank, as are gaps.

In [ ]:
cq.show_superposition_blocks(models, superpositions, ALIGNMENT_COLOUR_SCHEME)

---
## 8. Multiple sequence alignment depth

Optional. The AFDB MSA endpoint is currently under maintenance and returns HTTP 403, so the automatic path here is **untested against a working service**. Supply your own a3m through `USER_MSA_PATH` above if you have one; it is accepted only when its gap-stripped target sequence matches the AFDB sequence for that accession.

In [ ]:
cq.report_msa_status(USER_MSA_PATH)

---
## 9. Summary

In [ ]:
cq.report_summary(models, family, position, alignments)

---
## What this notebook cannot tell you

- **Whether another model is appropriate for your purpose.** It shows the family's range and its extremes with the evidence attached. It deliberately does not rank candidates or recommend a substitution: no single available number reliably separates a functionally divergent member from a legitimate distant homologue, so a recommendation would be one the notebook cannot stand behind.
- **Whether a protein will behave experimentally.** pLDDT is the predictor's certainty, nothing more.
- **Whether a low family mean means poor prediction.** It is substantially a function of disorder content. Read the per-residue profiles.

A later version may add family-level annotation, which would allow a screened candidate shortlist.